# Gen AI Project: 2
### NxtWave Academy | Build an HR chatbot using RAG

---

## Objective

Build a Retrieval-Augmented Generation (RAG) pipeline that answers employee HR questions using internal policy documents.

## What you will build

- Load and process HR policy documents
- Create chunks and embeddings
- Build a vector database using FAISS
- Implement a RAG pipeline with guardrails
- Generate your `submission.csv`


## Write your solution code Below

## Generate `submission.csv`

Generate your final `submission.csv` file for submission.

Do not modify this cell.


# install Dependancy 

In [98]:
import sys

print("Python being used:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python being used:
C:\ProgramData\anaconda3\python.exe

Python version:
3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]


In [99]:
import sys

!{sys.executable} -m pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface chromadb

Defaulting to user installation because normal site-packages is not writeable


In [100]:
print("Installing Dependencies")

%pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-groq \
    langchain-google-genai \
    langchain-openai \
    langchain-core \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    transformers \
    torch \
    huggingface-hub \
    groq \
    langsmith \
    python-dotenv \
    tiktoken

print("Installation completed")

Installing Dependencies
Note: you may need to restart the kernel to use updated packages.
Installation completed


In [101]:
import langchain
import langchain_community
import langchain_huggingface
import langchain_groq
import langchain_google_genai
import langchain_openai
import faiss
import pypdf
import transformers
import torch
import groq

In [102]:
import sys

print(sys.executable)

C:\ProgramData\anaconda3\python.exe


In [ ]:
LLM_PROVIDER = "groq"
LLM_MODEL = "openai/gpt-oss-20b"
LLM_API_KEY = "give the API key"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

CORPUS_PATH = "./zyro-dynamics-hr-corpus/"

In [104]:
import os

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate

print("All imports successful!")

All imports successful!


In [105]:
CURPUS_PATH = "./zyro-dynamics-hr-corpus/"
loader = PyPDFDirectoryLoader(CURPUS_PATH)
documents = loader.load()
print(f"Loaded{len(documents)}documents")

Loaded39documents


In [106]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 100
)
chunks = splitter.split_documents(documents)
print(f"Created{len(chunks)}chunks")

Created107chunks


# Embeddings

### Hugging Face Embeddings

In [107]:
from langchain_huggingface import HuggingFaceEmbeddings

In [108]:
model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## vectorDB + Retrival
### FAISS vector DB

In [109]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks,model) # chunks :- chunks of the data, model :- Embedding models

# Retrival
retriever = vectorstore.as_retriever(search_kwargs = {"k" : 3})
print("done")

done


# LLM Initialization

In [110]:
from langchain_groq import ChatGroq
LLM_MODEL = "openai/gpt-oss-20b"
llm_model = ChatGroq(
    model=LLM_MODEL,
    temperature=0.7,
    max_tokens=500,
    api_key=LLM_API_KEY
)
print("LLM Model initialized:", LLM_MODEL)
# from langchain_groq import ChatGroq
# llm_model = ChatGroq(
# model = 'groq',
# temperature = 0.7,
# max_tokens = 500,
# api_key = LLM_API_KEY
# )
# print(f"LLM Model groq initialized")

LLM Model initialized: openai/gpt-oss-20b


In [111]:
RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are an HR assistant.

Answer the question using ONLY the information
provided in the context.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. If the answer is present in the context,
   answer it clearly and directly.
4. If the answer is not present in the context,
   say:
   "I don't have that information in the provided HR documents."

Context:
{context}

Question:
{question}

Answer:
"""
)


# RAG Chain

In [112]:
from langsmith import traceable
from langchain_core.output_parsers import StrOutputParser
RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are an HR assistant.Answer the question using only
    the context below. if the answer isn't in the context,say 
    you don't have that information
    context : {context},
    Question : {question}"""
)
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)
@traceable(name="rag_chain")
def rag_chain(question: str):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    chain = RAG_PROMPT | llm_model | StrOutputParser()
    answer = chain.invoke({
        "context": context,
        "question": question
    })
    return {
        "answer": answer,
        "sources": docs
    }

In [113]:
result = rag_chain("What benefits are provided to employees?")
print("Answer:", result["answer"])
print("\nRetrieved from:")
for doc in result["sources"]:
    print(doc.metadata.get("sources"))

Answer: **Benefits provided to employees (as per the Zyro Dynamics Pvt. Ltd. Compensation and Benefits Policy)**  

| Benefit | Key details |
|---------|-------------|
| **Employee Assistance Programme (EAP)** | Up to 6 free, confidential counselling sessions per year through the Company’s EAP partner. |
| **Employee Stock Options (ESOP)** | Offered to employees at grade L5 and above; 4‑year vesting schedule with a 1‑year cliff. |
| **Health coverage for dependents** | All premiums for dependents are fully paid by the Company (specific plan details not provided). |
| **Personal Accident Insurance** | Coverage equivalent to 5 × the employee’s annual CTC. |
| **Term Life Insurance** | Coverage of 3 × the annual CTC for all permanent employees. |
| **Provident Fund (PF)** | Both employee and Company contribute 12 % of the employee’s basic salary each month to the PF account

Retrieved from:
None
None
None


In [114]:
print("=" * 50)
print("Submission Generator")
print("=" * 50)

print(f"\nGenerating responses for {len(eval_questions)} questions...\n")

rows = []

for i, q in enumerate(eval_questions):
    qid = q["question_id"]
    question = q["question"]

    try:
        result = ask_bot(question)
        answer = result["answer"]
        status = "OK"
    except Exception as e:
        answer = f"Error: {str(e)}"
        status = "ERROR"

    rows.append({
        "question_id": qid,
        "answer": answer,
    })

    print(f"[{i+1:02d}/{len(eval_questions)}] {qid} ... {status}")

    if i < len(eval_questions) - 1:
        time.sleep(2)

csv_path = "submission.csv"

fieldnames = ["question_id", "answer"]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("\nsubmission.csv generated successfully.")

Submission Generator


NameError: name 'eval_questions' is not defined